# Prototype Selection Strategy Ablation

Compare validation accuracy curves for **Centroid**, **Random**, and **Font** prototype selection strategies on EMNIST (balanced & digits splits). 5 seeds per (dataset, strategy) cell — plotted as mean line with min–max band.

In [ ]:
import re
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# --- Publication style (mirrors notebooks/04_misclassification_analysis.ipynb) ---
PUB_FONT = dict(family="Computer Modern, CMU Serif, Times New Roman, serif", size=12, color="black")

pio.templates["pub"] = go.layout.Template(
    layout=dict(
        font=PUB_FONT,
        paper_bgcolor="white",
        plot_bgcolor="white",
        colorway=[
            "#1f77b4", "#d62728", "#2ca02c", "#ff7f0e",
            "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
            "#bcbd22", "#17becf",
        ],
        xaxis=dict(
            showline=True, linewidth=1, linecolor="black", mirror=True,
            ticks="outside", tickwidth=1, tickcolor="black", ticklen=4,
            showgrid=False, zeroline=False, automargin=True,
        ),
        yaxis=dict(
            showline=True, linewidth=1, linecolor="black", mirror=True,
            ticks="outside", tickwidth=1, tickcolor="black", ticklen=4,
            showgrid=False, zeroline=False, automargin=True,
        ),
        legend=dict(bgcolor="rgba(255,255,255,0.9)", bordercolor="black", borderwidth=1),
        margin=dict(l=60, r=20, t=50, b=55),
    )
)
pio.templates.default = "pub"

PLOTLY_CONFIG = {
    "toImageButtonOptions": {"format": "svg", "filename": "figure", "scale": 1},
    "displaylogo": False,
    "modeBarButtonsToRemove": ["lasso2d", "select2d"],
}

print("Setup OK")

In [ ]:
# --- Load CSV and parse Run name into (dataset, strategy, seed) ---
CSV_PATH = Path("../results/prototype_vs_font_ablation/prototype_vs_font_ablation.csv")
df = pd.read_csv(CSV_PATH)

# Run name examples:
#   ssl_emnist_balanced_conv-seed3            -> dataset=balanced, strategy=Random
#   ssl_emnist_balanced_conv_centroid-seed3   -> dataset=balanced, strategy=Centroid
#   ssl_emnist_balanced_conv_font-seed3       -> dataset=balanced, strategy=Font
RUN_RE = re.compile(r"ssl_emnist_(balanced|digits)_conv(?:_(centroid|font))?-seed(\d+)")

def parse_run(name):
    m = RUN_RE.fullmatch(name)
    if not m:
        return pd.Series({"dataset": None, "strategy": None, "seed": None})
    ds, strat, seed = m.groups()
    strategy = {"centroid": "Centroid", "font": "Font", None: "Random"}[strat]
    return pd.Series({"dataset": ds, "strategy": strategy, "seed": int(seed)})

df = df.join(df["Run"].apply(parse_run))
assert df["strategy"].notna().all(), df.loc[df["strategy"].isna(), "Run"].unique()

print(df.groupby(["dataset", "strategy"])["seed"].nunique())
df.head()

In [ ]:
# --- Plot helper: mean line + min/max band per strategy for one dataset ---
STRATEGY_ORDER = ["Centroid", "Random", "Font"]
STRATEGY_COLORS = {
    "Centroid": "#1f77b4",  # tab10 blue
    "Random":   "#d62728",  # tab10 red
    "Font":     "#2ca02c",  # tab10 green
}

def _hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

def plot_ablation(df_ds, title, save_path=None):
    fig = go.Figure()
    for strat in STRATEGY_ORDER:
        sub = df_ds[df_ds["strategy"] == strat]
        if sub.empty:
            continue
        # All runs in a (dataset, strategy) group share the same step grid; map to epoch 1..N for the x-axis.
        agg = (
            sub.groupby("step")["value"]
               .agg(mean="mean", lo="min", hi="max")
               .reset_index()
               .sort_values("step")
        )
        epoch = list(range(1, len(agg) + 1))
        color = STRATEGY_COLORS[strat]
        band_color = _hex_to_rgba(color, 0.18)

        fig.add_trace(go.Scatter(
            x=epoch + epoch[::-1],
            y=list(agg["hi"]) + list(agg["lo"])[::-1],
            fill="toself", fillcolor=band_color,
            line=dict(color="rgba(0,0,0,0)"),
            hoverinfo="skip", showlegend=False, name=f"{strat} band",
        ))
        fig.add_trace(go.Scatter(
            x=epoch, y=list(agg["mean"]),
            mode="lines", name=strat,
            line=dict(color=color, width=2),
        ))

    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center"),
        xaxis_title="Epoch",
        yaxis_title="Validation accuracy",
        width=720, height=460,
        legend=dict(x=0.98, y=0.05, xanchor="right", yanchor="bottom"),
    )
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.write_image(str(save_path), format="svg")
        print(f"Saved {save_path}")
    return fig

In [ ]:
# --- EMNIST balanced ---
fig_bal = plot_ablation(
    df[df["dataset"] == "balanced"],
    title="Prototype selection \u2014 EMNIST balanced",
    save_path="../results/prototype_vs_font_ablation/prototype_vs_font_emnist_balanced.svg",
)
fig_bal.show(config=PLOTLY_CONFIG)

In [ ]:
# --- EMNIST digits ---
fig_dig = plot_ablation(
    df[df["dataset"] == "digits"],
    title="Prototype selection \u2014 EMNIST digits",
    save_path="../results/prototype_vs_font_ablation/prototype_vs_font_emnist_digits.svg",
)
fig_dig.show(config=PLOTLY_CONFIG)